In [1]:
import pandas as pd
import sqlite3
from google.colab import files

# Upload the Superstore CSV (the same one from your Power BI project)
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename, encoding='latin-1')

# Clean column names so SQL is easy (no spaces or hyphens)
df.columns = df.columns.str.replace(' ', '_').str.replace('-', '_')

# Load it into a SQLite database table called 'sales'
conn = sqlite3.connect('superstore.db')
df.to_sql('sales', conn, if_exists='replace', index=False)

# A small helper so we can run SQL and see results as a table
def run_query(q):
    return pd.read_sql_query(q, conn)

print("Done. Your table is called 'sales'. Columns:")
print(df.columns.tolist())


Saving SampleSuperstore.csv to SampleSuperstore.csv
Done. Your table is called 'sales'. Columns:
['Ship_Mode', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Category', 'Sub_Category', 'Sales', 'Quantity', 'Discount', 'Profit']


In [2]:
run_query("SELECT * FROM sales LIMIT 5")

,Ship_Mode,Segment,Country,City,State,Postal_Code,Region,Category,Sub_Category,Sales,Quantity,Discount,Profit
0,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.9600,2,0.00,41.9136
1,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.9400,3,0.00,219.5820
2,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,14.6200,2,0.00,6.8714
3,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,957.5775,5,0.45,-383.0310
4,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,22.3680,2,0.20,2.5164


In [3]:
run_query("SELECT Region, Sales, Profit FROM sales LIMIT 10")

,Region,Sales,Profit
0,South,261.9600,41.9136
1,South,731.9400,219.5820
2,West,14.6200,6.8714
3,South,957.5775,-383.0310
4,South,22.3680,2.5164
5,West,48.8600,14.1694
6,West,7.2800,1.9656
7,West,907.1520,90.7152
8,West,18.5040,5.7825
9,West,114.9000,34.4700


In [4]:
run_query("SELECT Region, Sales, Profit FROM sales WHERE Region = 'West'")

,Region,Sales,Profit
0,West,14.620,6.8714
1,West,48.860,14.1694
2,West,7.280,1.9656
3,West,907.152,90.7152
4,West,18.504,5.7825
...,...,...,...
3198,West,36.240,15.2208
3199,West,91.960,15.6332
3200,West,258.576,19.3932
3201,West,29.600,13.3200


In [9]:
run_query('''
SELECT Region,
       ROUND(SUM(Sales), 2) AS Total_Sales,
       ROUND(SUM(Profit), 2) AS Total_Profit
FROM sales
GROUP BY Region
ORDER BY Total_Sales DESC
''')

,Region,Total_Sales,Total_Profit
0,West,725457.82,108418.45
1,East,678781.24,91522.78
2,Central,501239.89,39706.36
3,South,391721.91,46749.43


In [10]:
run_query('''
SELECT Sub_Category, ROUND(SUM(Sales), 2) AS Total_Sales
FROM sales
GROUP BY Sub_Category
ORDER BY Total_Sales DESC
LIMIT 5
''')

,Sub_Category,Total_Sales
0,Phones,330007.05
1,Chairs,328449.10
2,Storage,223843.61
3,Tables,206965.53
4,Binders,203412.73


In [11]:
run_query('''
SELECT Sub_Category, ROUND(SUM(Profit), 2) AS Total_Profit
FROM sales
GROUP BY Sub_Category
HAVING SUM(Profit) < 0
ORDER BY Total_Profit
''')

,Sub_Category,Total_Profit
0,Tables,-17725.48
1,Bookcases,-3472.56
2,Supplies,-1189.10


In [12]:
run_query('''
SELECT Category,
       ROUND(SUM(Sales), 2) AS Total_Sales,
       ROUND(SUM(Profit), 2) AS Total_Profit,
       ROUND(100.0 * SUM(Profit) / SUM(Sales), 1) AS Margin_Pct
FROM sales
GROUP BY Category
ORDER BY Margin_Pct DESC
''')

,Category,Total_Sales,Total_Profit,Margin_Pct
0,Technology,836154.03,145454.95,17.4
1,Office Supplies,719047.03,122490.80,17.0
2,Furniture,741999.80,18451.27,2.5
